In [1]:
import numpy as np
import pandas as pd 
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder ,OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import cross_validate, KFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import SGDRegressor,LinearRegression,Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import VotingRegressor

In [2]:
df_train =pd.read_csv("/Users/thomasalaby/Desktop/ML-PRO/ML_PROJECT/flight_price_prediction/data/processed/train.csv",)
df_train.head(5)

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,78798,AirAsia,I5-942,Mumbai,Afternoon,one,Evening,Chennai,Economy,4.33,13,5939
1,42796,Air_India,AI-636,Delhi,Afternoon,one,Morning,Chennai,Economy,19.83,48,3988
2,118628,SpiceJet,SG-4009,Bangalore,Early_Morning,zero,Early_Morning,Chennai,Economy,1.08,38,1606
3,61731,Indigo,6E-354,Mumbai,Afternoon,two_or_more,Night,Kolkata,Economy,9.92,2,21483
4,25678,AirAsia,I5-747,Delhi,Evening,one,Late_Night,Kolkata,Economy,6.58,33,3175


In [3]:
y_train=df_train["price"]
X_train=df_train.drop("price",axis=1)

In [4]:
X_train.shape

(210107, 11)

In [5]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 210107 entries, 0 to 210106
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        210107 non-null  int64  
 1   airline           210107 non-null  str    
 2   flight            210107 non-null  str    
 3   source_city       210107 non-null  str    
 4   departure_time    210107 non-null  str    
 5   stops             210107 non-null  str    
 6   arrival_time      210107 non-null  str    
 7   destination_city  210107 non-null  str    
 8   class             210107 non-null  str    
 9   duration          210107 non-null  float64
 10  days_left         210107 non-null  int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 28.4 MB


Encodage simple de `stops`et `class``
 * `stops`: variable ordinale(zero,one,two_or_more)->0/1/2
 * `class`:Economy/Business -> 0/1

In [6]:
mapping_stops={"zero":0,"one":1,"two_or_more":2}
X_train['stops_num']=X_train["stops"].map(mapping_stops)
print(X_train[["stops","stops_num"]].head())

         stops  stops_num
0          one          1
1          one          1
2         zero          0
3  two_or_more          2
4          one          1


In [7]:
mapping_class={"Economy":0,"Business":1}
X_train["class_num"]=X_train["class"].map(mapping_class)
print(X_train[["class","class_num"]].head())

     class  class_num
0  Economy          0
1  Economy          0
2  Economy          0
3  Economy          0
4  Economy          0


In [8]:
X_train.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,stops_num,class_num
0,78798,AirAsia,I5-942,Mumbai,Afternoon,one,Evening,Chennai,Economy,4.33,13,1,0
1,42796,Air_India,AI-636,Delhi,Afternoon,one,Morning,Chennai,Economy,19.83,48,1,0
2,118628,SpiceJet,SG-4009,Bangalore,Early_Morning,zero,Early_Morning,Chennai,Economy,1.08,38,0,0
3,61731,Indigo,6E-354,Mumbai,Afternoon,two_or_more,Night,Kolkata,Economy,9.92,2,2,0
4,25678,AirAsia,I5-747,Delhi,Evening,one,Late_Night,Kolkata,Economy,6.58,33,1,0


In [9]:
X_train.drop(columns=["Unnamed: 0","flight","stops","class"])

,airline,source_city,departure_time,arrival_time,destination_city,duration,days_left,stops_num,class_num
0,AirAsia,Mumbai,Afternoon,Evening,Chennai,4.33,13,1,0
1,Air_India,Delhi,Afternoon,Morning,Chennai,19.83,48,1,0
2,SpiceJet,Bangalore,Early_Morning,Early_Morning,Chennai,1.08,38,0,0
3,Indigo,Mumbai,Afternoon,Night,Kolkata,9.92,2,2,0
4,AirAsia,Delhi,Evening,Late_Night,Kolkata,6.58,33,1,0
...,...,...,...,...,...,...,...,...,...
210102,Air_India,Kolkata,Afternoon,Night,Hyderabad,8.08,22,1,0
210103,AirAsia,Kolkata,Early_Morning,Evening,Bangalore,9.42,15,1,0
210104,Vistara,Mumbai,Early_Morning,Night,Kolkata,16.25,39,1,1
210105,Vistara,Delhi,Early_Morning,Morning,Bangalore,5.00,8,1,1


In [10]:
numerical_features=["duration","days_left","stops_num","class_num"]
categorical_features=["airline","source_city","departure_time","arrival_time","destination_city"]

In [11]:
# Transformation des données numeriques 
numeric_transformer = Pipeline(steps=[('imputer',SimpleImputer(strategy='median')),
                                      ('scaler',StandardScaler())
                                      ])

In [12]:
# Transformation des données catégorielles
categorical_transformer = Pipeline(steps=[('imputer' ,SimpleImputer(strategy='constant',fill_value='missing')),
                                  ('onehot', OneHotEncoder(handle_unknown='ignore',drop='first'))
                                  ])

In [13]:
preprocessor = ColumnTransformer(transformers=[('num',numeric_transformer,numerical_features),
                                           ('cat',categorical_transformer,categorical_features)
                                           ])
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

-----

## **Modélisation**


In [14]:
# fonction utiliotaire pour évaluer un modèle
def evaluate_model(
    name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    result_dict,
    cv=5,
):
    """
    Évalue un modèle avec validation croisée sur le train,
    puis refit sur tout le train et évalue une fois sur le test.

    Paramètres
    ----------
    name : str
        Nom du modèle (pour l'affichage et le dictionnaire de résultats).
    model : estimator / Pipeline
        Modèle ou Pipeline scikit-learn (déjà construit).
    X_train, y_train : array-like
        Données d'entraînement.
    X_test, y_test : array-like
        Données de test.
    results_dict : dict
        Dictionnaire dans lequel on stocke les résultats.
    cv : int
        Nombre de folds pour la validation croisée.
    """
    # ---- 1) Validation croisée sur le train set ----
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=KFold(n_splits=cv, shuffle=True, random_state=42),
        scoring={
            "mae":"neg_mean_absolute_error",
            "rmse":"neg_root_mean_squared_error",
            "r2":"r2",
        },
    )
    # moyennes MAE/RMSE
    cv_mae_mean = -cv_results["test_mae"].mean()
    cv_mae_std =  cv_results["test_mae"].std()
    
    cv_rmse_mean=-cv_results["test_rmse"].mean()
    cv_rmse_std= cv_results["test_rmse"].std()
    
    cv_r2_mean = cv_results["test_r2"].mean()
    cv_r2_std = cv_results["test_r2"].std()
    
    print(f"=== {name} ===")
    print(f"Validation croisée ({cv}-fold sur le train:)")
    print(f" MAE (mean ± std ) : {cv_mae_mean:.2f} ± {cv_mae_std:.2f}")
    print(f"RMSE (mean ± std ) : {cv_rmse_mean:.2f} ± {cv_rmse_std:.2f}")
    print(f"r2 (mean ± std): {cv_r2_mean:.4f} ± {cv_r2_std:.4f}")
    # ----- 2) Refit sur tout le train---
    model.fit(X_train, y_train)
    # ----- 3) Évaluation finale sur le test ----
    y_test_pred = model.predict(X_test)
    
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = root_mean_squared_error(y_test,y_test_pred)
    test_r2 = r2_score(y_test,y_test_pred)
    print("Évaluation finale sur test:")
    print(f" Test MAE : {test_mae:.2f}")
    print(f" Test RMSE : {test_rmse:.2f}")
    print(f" Test R² : {test_r2:.4f}")
    
    result_dict[name]={
        "cv_mae_mean":cv_mae_mean,
        "cv_mae_std":cv_mae_std,
        "cv_rmse_mean":cv_rmse_mean,
        "cv_rmse_std":cv_rmse_std,
        "cv_r2_mean":cv_r2_mean,
        "cv_r2_std":cv_r2_std,
        "test_mae":test_mae,
        "test_rmse":test_rmse,
        "test_r2":test_r2,
    }

In [15]:
# Dictionnnaire pour stocker les resultats 
results = {}

In [16]:
df_test = pd.read_csv("/Users/thomasalaby/Desktop/ML-PRO/ML_PROJECT/flight_price_prediction/data/processed/test.csv")
df_test.head()

,Unnamed: 0.1,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,27131,27131,Air_India,AI-506,Delhi,Morning,one,Early_Morning,Kolkata,Economy,19.75,40,7366
1,266857,266857,Vistara,UK-706,Kolkata,Morning,one,Night,Mumbai,Business,9.83,42,64831
2,141228,141228,Vistara,UK-772,Kolkata,Morning,one,Night,Bangalore,Economy,10.50,41,6195
3,288329,288329,Vistara,UK-824,Chennai,Night,one,Morning,Delhi,Business,14.50,14,60160
4,97334,97334,Air_India,AI-501,Bangalore,Afternoon,one,Night,Mumbai,Economy,8.25,20,6578


In [17]:
mapping_stops={"zero":0,"one":1,"two_or_more":2}
df_test['stops_num']=df_test["stops"].map(mapping_stops)
print(df_test[["stops","stops_num"]].head())

  stops  stops_num
0   one          1
1   one          1
2   one          1
3   one          1
4   one          1


In [18]:
mapping_class={"Economy":0,"Business":1}
df_test["class_num"]=df_test["class"].map(mapping_class)
print(df_test[["class","class_num"]].head())

      class  class_num
0   Economy          0
1  Business          1
2   Economy          0
3  Business          1
4   Economy          0


In [19]:
df_test.head()

,Unnamed: 0.1,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,stops_num,class_num
0,27131,27131,Air_India,AI-506,Delhi,Morning,one,Early_Morning,Kolkata,Economy,19.75,40,7366,1,0
1,266857,266857,Vistara,UK-706,Kolkata,Morning,one,Night,Mumbai,Business,9.83,42,64831,1,1
2,141228,141228,Vistara,UK-772,Kolkata,Morning,one,Night,Bangalore,Economy,10.50,41,6195,1,0
3,288329,288329,Vistara,UK-824,Chennai,Night,one,Morning,Delhi,Business,14.50,14,60160,1,1
4,97334,97334,Air_India,AI-501,Bangalore,Afternoon,one,Night,Mumbai,Economy,8.25,20,6578,1,0


In [20]:
df_test.drop(columns=["Unnamed: 0.1","Unnamed: 0","flight","stops","class"])

,airline,source_city,departure_time,arrival_time,destination_city,duration,days_left,price,stops_num,class_num
0,Air_India,Delhi,Morning,Early_Morning,Kolkata,19.75,40,7366,1,0
1,Vistara,Kolkata,Morning,Night,Mumbai,9.83,42,64831,1,1
2,Vistara,Kolkata,Morning,Night,Bangalore,10.50,41,6195,1,0
3,Vistara,Chennai,Night,Morning,Delhi,14.50,14,60160,1,1
4,Air_India,Bangalore,Afternoon,Night,Mumbai,8.25,20,6578,1,0
...,...,...,...,...,...,...,...,...,...,...
45018,Air_India,Bangalore,Afternoon,Evening,Hyderabad,28.25,29,10090,1,0
45019,Indigo,Kolkata,Early_Morning,Afternoon,Bangalore,6.50,26,4284,1,0
45020,Vistara,Mumbai,Early_Morning,Morning,Delhi,2.17,34,36700,0,1
45021,Vistara,Chennai,Night,Night,Mumbai,24.50,9,20078,1,0


In [21]:
X_test=df_test.drop("price",axis=1)
y_test=df_test["price"]

### **1.Baseline DummyRegressor**

In [22]:
dummy_model = DummyRegressor( strategy='mean')
evaluate_model(
    name= "DummyRegressor (mean)",
    model=dummy_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    result_dict=results,
)

=== DummyRegressor (mean) ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 19769.13 ± 25.42
RMSE (mean ± std ) : 22702.11 ± 33.35
r2 (mean ± std): -0.0000 ± 0.0000
Évaluation finale sur test:
 Test MAE : 19740.62
 Test RMSE : 22658.89
 Test R² : -0.0000


### **2.Régression linéaire `LinearRegression`**

In [23]:

linreg_pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("linear_regression",LinearRegression())
])

evaluate_model(
    name= "LinearRegression",
    model=linreg_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    result_dict=results,
)

=== LinearRegression ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 4527.28 ± 21.35
RMSE (mean ± std ) : 6813.65 ± 45.49
r2 (mean ± std): 0.9099 ± 0.0012
Évaluation finale sur test:
 Test MAE : 4506.94
 Test RMSE : 6813.94
 Test R² : 0.9096


### **3.Régression linéaire `Ridge`**

In [24]:
alphas=[0.01,0.1,1.0]
for a in alphas :
    ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=a))
])
    evaluate_model(
    name= f"ridge (alpha={a})",
    model=ridge_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    result_dict=results,
) 
    
    


=== ridge (alpha=0.01) ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 4527.28 ± 21.36
RMSE (mean ± std ) : 6813.66 ± 45.49
r2 (mean ± std): 0.9099 ± 0.0012
Évaluation finale sur test:
 Test MAE : 4506.96
 Test RMSE : 6813.99
 Test R² : 0.9096
=== ridge (alpha=0.1) ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 4527.28 ± 21.36
RMSE (mean ± std ) : 6813.66 ± 45.49
r2 (mean ± std): 0.9099 ± 0.0012
Évaluation finale sur test:
 Test MAE : 4506.94
 Test RMSE : 6813.99
 Test R² : 0.9096
=== ridge (alpha=1.0) ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 4527.26 ± 21.36
RMSE (mean ± std ) : 6813.66 ± 45.49
r2 (mean ± std): 0.9099 ± 0.0012
Évaluation finale sur test:
 Test MAE : 4506.86
 Test RMSE : 6814.00
 Test R² : 0.9096


### **4. Descente de gradiente `SGDRegressor`**

In [25]:
sgdreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("sgdregressor", SGDRegressor(alpha=0.1,learning_rate='constant'))
])
evaluate_model(
    name= "sgdregressor",
    model=sgdreg_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    result_dict=results,
)

=== sgdregressor ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 4811.97 ± 200.36
RMSE (mean ± std ) : 7283.24 ± 249.80
r2 (mean ± std): 0.8969 ± 0.0073
Évaluation finale sur test:
 Test MAE : 4870.62
 Test RMSE : 7461.25
 Test R² : 0.8916


### **5. DecisssionTreeRegressor**


In [26]:
decision_tree_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("decisiontree",DecisionTreeRegressor() )
])
evaluate_model(
    name= "decision_tree_reg",
    model=decision_tree_reg,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    result_dict=results,
)

=== decision_tree_reg ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 1228.10 ± 11.03
RMSE (mean ± std ) : 3555.64 ± 35.89
r2 (mean ± std): 0.9755 ± 0.0005
Évaluation finale sur test:
 Test MAE : 1195.14
 Test RMSE : 3535.91
 Test R² : 0.9756


### **6.Foret aleatoire `RandomForestRegressor`**

In [27]:
ranforeg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("ranforeg",RandomForestRegressor(n_estimators=50, n_jobs=-1) )
])
evaluate_model(
    name= "RandomForestRegressor",
    model=ranforeg,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    result_dict=results,
)

=== RandomForestRegressor ===
Validation croisée (5-fold sur le train:)
 MAE (mean ± std ) : 1142.95 ± 8.29
RMSE (mean ± std ) : 2817.09 ± 33.01
r2 (mean ± std): 0.9846 ± 0.0004
Évaluation finale sur test:
 Test MAE : 1108.43
 Test RMSE : 2795.51
 Test R² : 0.9848


## **Comparaison des modèles**

In [28]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by='cv_mae_mean')
results_df

,cv_mae_mean,cv_mae_std,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_r2_std,test_mae,test_rmse,test_r2
RandomForestRegressor,1142.951136,8.286895,2817.085498,33.009043,0.984599,0.000359,1108.425797,2795.509281,0.984779
decision_tree_reg,1228.099195,11.031006,3555.643610,35.892293,0.975466,0.000500,1195.143815,3535.912968,0.975648
ridge (alpha=1.0),4527.264155,21.360669,6813.656170,45.486224,0.909913,0.001231,4506.860384,6813.995808,0.909566
ridge (alpha=0.1),4527.277423,21.358687,6813.656678,45.486987,0.909913,0.001232,4506.942526,6813.989365,0.909567
ridge (alpha=0.01),4527.277948,21.363531,6813.655991,45.487647,0.909913,0.001232,4506.955820,6813.986144,0.909567
LinearRegression,4527.282364,21.351439,6813.651841,45.487308,0.909913,0.001232,4506.940156,6813.944196,0.909568
sgdregressor,4811.969359,200.360500,7283.238219,249.801165,0.896942,0.007342,4870.618004,7461.254197,0.891570
DummyRegressor (mean),19769.133102,25.420591,22702.114373,33.353209,-0.000028,0.000024,19740.616691,22658.886810,-0.000007


## **Modèle final**

In [29]:
best_model = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ('model', RandomForestRegressor(n_estimators=100,
                                    max_depth=None,
                                    min_samples_leaf=1,
                                    ))
])
best_model.fit(X_train,  y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['Unnamed: 0','airline','flight',...,'days_left','stops_num','class_num']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of colum

## **Évaluation finale du modèle**

In [30]:
y_test_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test,y_test_pred)
rmse= root_mean_squared_error(y_test,y_test_pred)
r2= r2_score(y_test,y_test_pred)

In [31]:
print(f"Test MAE: {mae:.2f}")
print(f"Test RMSE: {rmse:.2f}")
print(f"test R²: {r2:.4f}")

Test MAE: 1105.26
Test RMSE: 2794.89
test R²: 0.9848


## **Intervalle de confiance sur la MAE**

In [32]:
# Prédictions 
y_test_pred = best_model.predict(X_test)
# Erreurs absolues (une par observation)
abs_errors =np.abs(y_test.to_numpy()-y_test_pred)

# MAE = moyenne des erreurs absolues 
mae = abs_errors.mean()
# Écart-type (std) des erreurs absolues 
sigma = abs_errors.std(ddof=1) # ddof=1 -> écart-type "échantillon"

# Erreur standard (SE)
n= len(abs_errors)
se= sigma / np.sqrt(n)

# z-score (95%=1.96)
z=1.96

# Intervalle de confiance 
ci_low= mae - z*se
ci_high = mae + z*se

print(f"n= {n}")
print(f"MAE = {mae:.2f}")
print(f"std(|erreur|) = {sigma:.2f}")
print(f"SE = {se:.4f}")
print(f"IC 95%: [{ci_low:.2f}, {ci_high:.2f}]")

n= 45023
MAE = 1105.26
std(|erreur|) = 2567.09
SE = 12.0983
IC 95%: [1081.55, 1128.97]


sur l'échantillon de test , la MAE vaut 1104.90 INR . En construisant un intervalle de confiance à 95% suivant la formule :
$$[MAE - z.SE;MAE +z.SE]$$
on obtient:
$$[1081.22, 1128.58]$$

Cela indique que, compte tenu de la taille finie du test, l'erreur moyenne "vraie" attendue en généralisation devrait se situer dans cette fourchette, avec une incertitude d'environ ±23.67 INR autour de la MAE observée. 
L'objectif est atteint:la MAE estimée est de tout au plus 16$ CAD et l'intervalle de confiance à 95% reste entierrement sous le seuil de 25$  CAD.  
En dollars canadien  on a :
$$[15.91;16.60]$$

## **Export du modèle final avec le préprocesseur intégré dans une pipeline**

In [33]:
import os

In [34]:
os.chdir("/Users/thomasalaby/Desktop/ML-PRO/ML_PROJECT/flight_price_Prediction")

In [35]:

import joblib
from pathlib import Path
from datetime import datetime
import json

# best_model est déjà une Pipeline scikit-learn : ("preprocessor" -> ColumnTransformer) + ("model" -> RandomForestRegressor)
# et il est déjà fit sur X_train, y_train

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)

MODEL_PATH = ARTIFACT_DIR / "flight_price_model.joblib"
META_PATH  = ARTIFACT_DIR / "flight_price_model.meta.json"

joblib.dump(best_model, MODEL_PATH)

metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "artifact": str(MODEL_PATH),
    "target": "price",
    "expected_features": [
        "airline", "source_city", "departure_time", "arrival_time", "destination_city",
        "duration", "days_left", "stops_num", "class_num"
    ],
    "notes": "Pipeline scikit-learn (preprocessor + model). Utiliser .predict() directement."
}
META_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Modèle exporté :", MODEL_PATH)
print("Métadonnées :", META_PATH)


Modèle exporté : artifacts/flight_price_model.joblib
Métadonnées : artifacts/flight_price_model.meta.json
